In [1]:
from diffusion_policy.dataset.tcl_dataset import TCLImageDataset
import mediapy
import numpy as np
import torch
from PIL import Image
from tqdm import tqdm
from robokit.debug_utils.images import concatenate_rgb_images, plot_action_wrt_time
from robokit.debug_utils.io import dataloader_speed_test

/mnt/dongxu-fs1/data-ssd/geyuan/programs/anaconda3/envs/robodiff/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")


In [14]:
dataset = TCLImageDataset(
    data_root="/home/geyuan/local_soft/TCL/0627_pot_object",
    h5_path="/home/geyuan/local_soft/TCL/hdf5/0627_pot_object_240p.h5",
    use_h5=True,
    horizon=16, pad_before=0, pad_after=7,
    shape_meta={
        "obs": {
            "image": {
                "shape": [3, 240, 320],
                "type": "rgb"
            },
            "gripper": {
                "shape": [3, 240, 320],
                "type": "rgb"
            },
            "joint_state": {
                "shape": [6],
                "type": "low_dim"
            }
        },
        "action": {
            "shape": [7,]
        }
    },
    transform_color_jitter=True,
)

[TCLDataset] loaded key=rel_actions shape=(10243, 7) from /home/geyuan/local_soft/TCL/0627_pot_object/extracted/rel_actions.npy
[TCLDataset] total length: 10243
[TCLDatasetHDF5] using h5 data: /home/geyuan/local_soft/TCL/hdf5/0627_pot_object_240p.h5
[TCLDataset] loading dataset statistics from: /home/geyuan/local_soft/TCL/0627_pot_object/statistics.json
[TCLImageDataset] dataset loaded, action_min=[-0.09999695 -0.09137268 -0.09607544 -0.47479459 -0.3806252  -0.64872884
  0.        ], action_max=[0.09529419 0.09685974 0.1        0.44823636 0.39019474 0.98823434
 1.        ]


In [10]:
dataloader_speed_test(dataset, num_workers=48)

workers=48, batch=64:   0%|                               | 1/1114 [00:03<1:11:05,  3.83s/it]

batch_data@0: Dict,keys=dict_keys(['obs', 'action'])
obs: Dict,keys=dict_keys(['image', 'gripper', 'joint_state'])
-image,<class 'torch.Tensor'>,shape=torch.Size([64, 1, 3, 240, 320])
-gripper,<class 'torch.Tensor'>,shape=torch.Size([64, 1, 3, 240, 320])
-joint_state,<class 'torch.Tensor'>,shape=torch.Size([64, 1, 6])
action,<class 'torch.Tensor'>,shape=torch.Size([64, 16, 7])


workers=48, batch=64:  90%|███████████████████████████▊   | 999/1114 [00:57<00:06, 17.44it/s]

workers=48, batch=64 ⇒ Samples=1000, Time=57.276s, Throughput=17.5 batches/s


In [9]:
global_idx = 0

task_skip_idx = 3
fps = 30

# task_skip_idx = 35
# fps = 5

print("Total tasks:", len(dataset.tcl_dataset.tasks))

for task_idx, task in enumerate(dataset.tcl_dataset.tasks):
    task_length = dataset.tcl_dataset.task_lengths[task_idx]
    images_primary, images_gripper = [], []
    images_cat = []
    actions = []

    if task_idx < task_skip_idx:
        global_idx += task_length
        continue

    for frame_idx in tqdm(range(task_length)):
        frame_data = dataset.tcl_dataset[global_idx]
        global_idx += 1
        images_primary.append(frame_data['primary_rgb'])
        images_gripper.append(frame_data['gripper_rgb'])
        images_cat.append(concatenate_rgb_images(frame_data['primary_rgb'], frame_data['gripper_rgb'], vertical=True, resize_ratio=1))
        actions.append(frame_data['rel_actions'])

    all_vis = []
    actions_vis, fig, ax = plot_action_wrt_time(np.array(actions))
    for frame_idx in range(task_length):
        all_vis.append(concatenate_rgb_images(images_cat[frame_idx], actions_vis[frame_idx],
                                              vertical=False, resize_ratio=1))

    mediapy.show_video(all_vis, fps=fps)

    break

Total tasks: 16


100%|███████████████████████████████████████████████████████████████████████| 590/590 [00:02<00:00, 227.44it/s]


Plotting action dynamic figures...


This browser does not support the video tag.


In [18]:
batch_data = dataset[104]

obs_image = []
obs_gripper = []
obs_joint_state = []
action = []

obs_image_data = batch_data['obs']['image']  # (T,C,H,W), in [-1,1]
obs_gripper_data = batch_data['obs']['gripper']
obs_image_data = (obs_image_data.permute(0, 2, 3, 1).numpy() * 127.5 + 127.5).astype(np.uint8)
obs_gripper_data = (obs_gripper_data.permute(0, 2, 3, 1).numpy() * 127.5 + 127.5).astype(np.uint8)
for obs_image_frame in obs_image_data:
    obs_image.append(obs_image_frame)
for obs_gripper_frame in obs_gripper_data:
    obs_gripper.append(obs_gripper_frame)
mediapy.show_video(obs_image, fps=5)
mediapy.show_video(obs_gripper, fps=5)

action_data = batch_data['action']  # (T,7)
for action_frame in action_data:
    action.append(action_frame)  # each is (7)
actions_vis, fig, ax = plot_action_wrt_time(np.array(action))
mediapy.show_video(actions_vis, fps=5)



Plotting action dynamic figures...


This browser does not support the video tag.
